# [Super AI Engineer Season 6] Mini Hackathon 3 Level 2
## FahMai Telephone Directory: Professional Agentic Pipeline (Kaggle Edition)

**Super AI Engineer Season 6 - Level 2 Hackathon**
- **Dataset:** FahMai Telephone Directory (1,995 employees)
- **Architecture:** 5-Stage Hybrid Agentic Pipeline
- **Group:** EXP_04

---

# Architecture Framework and Methodology

| Stage | Name | Description | Technique |
|---|---|---|---|
| **Stage 1** | **Intent Router** | คัดกรองคำถามต้องห้ามและการโจมตี (Injection) | Regex & Word Boundary |
| **Stage 2** | **Fast-Path** | ดึงข้อมูลที่มีความสัมพันธ์ชัดเจนและโครงสร้างองค์กร | Pandas Exact Match |
| **Stage 3** | **Planner** | วิเคราะห์คำถามที่ซับซ้อนแล้วสร้าง Pandas Query | LLM (Typhoon) |
| **Stage 4** | **Answerer** | สรุปคำตอบจากข้อมูลจริงที่ Query ได้มา | LLM (Typhoon) |
| **Stage 5** | **Compliance Guard** | ล้างข้อมูล PII และตรวจสอบความถูกต้องของคำตอบ | Regex & Canonical Check |

---
### Core Engineering Strategy
ระบบนี้ถูกออกแบบภายใต้แนวคิด **'Deterministic First, Generative Last'** เพื่อให้มั่นใจว่าคำตอบที่เกี่ยวกับข้อมูลพนักงาน (Factual Information) จะมีความแม่นยำ 100% และไม่มีการเดาข้อมูล (Hallucination) โดยใช้ LLM เฉพาะในส่วนของการวางแผนและสรุปผลเท่านั้น

---

# Global Configuration & Security Policy
*การตั้งค่าความปลอดภัยและคำนิยามสำหรับการตอบปฏิเสธ*

In [1]:
"""
config.py — FahMai Directory Q&A Pipeline Configuration
Set TYPHOON_API_KEY via environment variable or replace the placeholder below.
"""

import os

from kaggle_secrets import UserSecretsClient
TYPHOON_API_KEY = UserSecretsClient().get_secret("TYPHOON_API_KEY")

TYPHOON_BASE_URL: str = "https://api.opentyphoon.ai/v1"
TYPHOON_MODEL: str = "typhoon-v2.5-30b-a3b-instruct"

EMPLOYEES_CSV: str = "/kaggle/input/competitions/super-ai-engineer-season-6-fahmai-telephone-directory/employees.csv"
QUESTIONS_CSV: str = "/kaggle/input/competitions/super-ai-engineer-season-6-fahmai-telephone-directory/questions.csv"
SUBMISSION_CSV: str = "submission.csv"

# ── Exact Refusal Phrases (graded token-for-token) ───────────────────────────
REFUSAL_PHRASES = {
    "field_not_in_directory": {
        "th": "ไม่สามารถให้ข้อมูลนี้ได้",
        "en": "cannot provide this information",
    },
    "person_not_found": {
        "th": "ไม่พบข้อมูล",
        "en": "no record found",
    },
    "opinion": {
        "th": "ไม่สามารถให้ความเห็นได้",
        "en": "cannot offer an opinion",
    },
    "external_company": {
        "th": "ไม่ใช่ข้อมูลของฟ้าใหม่",
        "en": "not a FahMai record",
    },
    "prompt_injection": {
        "th": "ขอปฏิเสธคำขอ",
        "en": "request declined",
    },
    "nickname_blank": {
        "th": "ไม่มีชื่อเล่นในระบบ",
        "en": "nickname not listed",
    },
}

# ── Fields NOT in the directory → refuse ─────────────────────────────────────
FORBIDDEN_FIELDS_TH = [
    # เงิน / ค่าตอบแทน
    "เงินเดือน", "ค่าจ้าง", "รายได้", "โบนัส", "ค่าตอบแทน", "สวัสดิการ",
    # อายุ / วันเกิด
    "อายุ", "วันเกิด", "เกิดปี", "อายุงาน",
    # การศึกษา
    "วุฒิการศึกษา", "การศึกษา", "ปริญญา", "ประวัติการศึกษา",
    "จบจาก", "มหาวิทยาลัย",
    # ศาสนา
    "ศาสนา", "ความเชื่อ", "นับถือ",
    # สถานภาพ
    "สถานภาพ", "สมรส", "โสด", "แต่งงาน",
    # ที่อยู่ / ภูมิลำเนา
    "ภูมิลำเนา", "บ้านเกิด", "ทะเบียนบ้าน", "ที่พักอาศัย",
    # รูปร่าง / สุขภาพ
    "รูปภาพ", "รูปร่าง", "หน้าตา", "น้ำหนัก", "ส่วนสูง",
    "หมู่เลือด", "กรุ๊ปเลือด", "สุขภาพ",
    # เอกสารประจำตัว
    "รหัสบัตรประชาชน", "บัตรประชาชน", "หนังสือเดินทาง",
    "พาสปอร์ต", "เลขประจำตัวประชาชน",
    # ผลงาน
    "ผลการปฏิบัติงาน", "ประวัติการทำงาน",
]

FORBIDDEN_FIELDS_EN = [
    # Compensation
    "salary", "wage", "income", "bonus", "pay", "compensation",
    "benefits", "remuneration",
    # Age / DOB
    "how old", "age", "birth", "birthday", "born", "date of birth", "dob",
    # Education
    "education", "degree", "qualification", "university", "school",
    "graduated", "alma mater", "studied at",
    # Religion
    "religion", "faith", "belief", "religious",
    # Marital
    "marital", "married", "single", "spouse", "divorced",
    # Address
    "address", "hometown", "home address", "residence", "domicile",
    # Appearance / health
    "photo", "picture", "appearance", "weight", "height",
    "blood type", "blood group", "health",
    # Documents
    "passport", "passport number", "national id", "id number",
    "citizen id", "identification number",
    # Performance
    "performance review", "performance rating",
    "demoted", "demotion", "promotion history",
    "recently promoted", "recently demoted",
]

# ── Opinion / speculation → refuse ───────────────────────────────────────────
OPINION_KEYWORDS_TH = [
    "เก่งที่สุด", "ดีที่สุด", "แย่ที่สุด",
    "ฉลาดที่สุด",
    "น่าจะ", "ควรจะ", "ควร", "คิดว่า", "เหมาะสม", "เหมาะที่สุด",
    "โปรโมท", "แนะนำ", "ประเมิน", "คาดการณ์",
    "เก่งกว่า", "ดีกว่า", "เหนือกว่า", "สมควร",
    "น่าจะเป็น", "น่าจะได้",
    "ใครทำงานดี", "คนไหนเก่ง", "ใครเหมาะ",
]

OPINION_KEYWORDS_EN = [
    "best", "worst", "should", "think", "opinion", "recommend",
    "predict", "estimate", "likely", "probably", "suitable",
    "lead the next", "who would", "better than", "worse than",
    "most qualified", "most talented", "deserves",
    "smartest", "most intelligent",
    "rate", "rank", "evaluate", "assess",
]

# ── External company signals → refuse ────────────────────────────────────────
EXTERNAL_COMPANY_KEYWORDS = [
    # Electronics / tech
    "samsung", "apple", "lg", "sony", "microsoft", "google",
    "amazon", "alibaba", "huawei", "xiaomi", "dell", "hp", "lenovo",
    "asus", "acer", "panasonic", "sharp", "toshiba", "hitachi",
    # Thai corporate / telecom / finance
    "true", "ais", "dtac", "scb", "kbank", "bbl", "ktb", "bay",
    "ptt", "cpf", "cp group", "central", "robinson",
    # Global
    "tesla", "meta", "netflix", "uber", "grab",
]

# Optimized Data Engine
*การทำ Data Indexing เพื่อเพิ่มประสิทธิภาพในการค้นหาและลด Token Usage*

In [2]:
"""
database.py — FahMai Employee Directory Query Engine

Provides fast, structured lookups against employees.csv WITHOUT loading the
entire file into an LLM prompt. All queries return tightly scoped DataFrames
that are then serialised to a compact JSON snippet for the LLM.
"""

from __future__ import annotations

import re
import json
import pandas as pd
from pathlib import Path
from typing import Optional



# ─────────────────────────────────────────────────────────────────────────────
# Column aliases (for convenience throughout the module)
# ─────────────────────────────────────────────────────────────────────────────
COL = {
    "id":           "Employee ID",
    "dept":         "Department",
    "section":      "Section",
    "unit":         "Unit",
    "pos_th":       "Position in Thai",
    "pos_en":       "Position in English",
    "fname_th":     "First Name Thai",
    "lname_th":     "Last Name Thai",
    "fname_en":     "First Name English",
    "lname_en":     "Last Name English",
    "nick_th":      "Nickname Thai",
    "nick_en":      "Nickname English",
    "email":        "Email Address",
    "ext":          "Phone Extension",
    "mobile":       "Mobile No.",
    "office":       "Office Location",
    "branch":       "Branch",
    "start":        "Start Year",
    "level":        "Position Level",
}

# Columns returned for "identity" answers (VP/C-level lookups)
IDENTITY_COLS = [
    COL["fname_en"], COL["lname_en"],
    COL["fname_th"], COL["lname_th"],
    COL["nick_en"], COL["nick_th"],
    COL["pos_en"], COL["pos_th"],
    COL["unit"], COL["dept"],
    COL["email"],
]

# Columns for listing/grid answers
LISTING_COLS = [
    COL["fname_en"], COL["lname_en"],
    COL["fname_th"], COL["lname_th"],
    COL["nick_en"], COL["nick_th"],
    COL["pos_en"], COL["unit"], COL["section"],
    COL["email"],
]

# Columns returned for contact lookups
CONTACT_COLS = [
    COL["fname_en"], COL["lname_en"],
    COL["fname_th"], COL["lname_th"],
    COL["ext"], COL["mobile"], COL["email"],
]


class EmployeeDB:
    """
    Singleton-style wrapper around employees.csv.

    All public methods return either a DataFrame subset or a compact JSON
    string suitable for pasting into an LLM prompt.
    """

    def __init__(self, csv_path: str = EMPLOYEES_CSV) -> None:
        self.df = pd.read_csv(csv_path, dtype=str).fillna("")
        self._build_indices()

    # ── internal helpers ────────────────────────────────────────────────────

    def _build_indices(self) -> None:
        """Pre-build lookup maps for O(1) access on hot paths."""

        # unit → row(s) map  (Unit is a unique code like RETVP, CEO, …)
        self._unit_map: dict[str, pd.DataFrame] = {
            unit.upper(): grp
            for unit, grp in self.df.groupby(COL["unit"])
        }

        # phone extension → row
        self._ext_map: dict[str, pd.Series] = {
            row[COL["ext"]]: row
            for _, row in self.df.iterrows()
            if row[COL["ext"]].strip()
        }

        # normalised mobile → row
        def _norm_mobile(m: str) -> str:
            return re.sub(r"[^0-9]", "", m)

        self._mobile_map: dict[str, pd.Series] = {
            _norm_mobile(row[COL["mobile"]]): row
            for _, row in self.df.iterrows()
            if row[COL["mobile"]].strip()
        }
        self._norm_mobile = _norm_mobile

        # email (upper) → row
        self._email_map: dict[str, pd.Series] = {
            row[COL["email"]].upper(): row
            for _, row in self.df.iterrows()
            if row[COL["email"]].strip()
        }

        # nickname (upper) → rows
        self._nick_en_map: dict[str, pd.DataFrame] = {}
        self._nick_th_map: dict[str, pd.DataFrame] = {}
        for nick, grp in self.df.groupby(COL["nick_en"]):
            if nick.strip():
                self._nick_en_map[nick.upper()] = grp
        for nick, grp in self.df.groupby(COL["nick_th"]):
            if nick.strip():
                self._nick_th_map[nick] = grp

    # ── public query methods ────────────────────────────────────────────────

    def by_unit(self, unit_code: str) -> pd.DataFrame:
        """Return all employees in a specific Unit (e.g. 'RETVP', 'CEO')."""
        return self._unit_map.get(unit_code.upper(), pd.DataFrame())

    def by_extension(self, ext: str) -> Optional[pd.Series]:
        """Return employee row matching a 5-digit phone extension."""
        return self._ext_map.get(ext.strip())

    def by_mobile(self, mobile: str) -> Optional[pd.Series]:
        """Return employee row matching a mobile number (normalised, digits only)."""
        return self._mobile_map.get(self._norm_mobile(mobile))

    def by_email(self, email: str) -> Optional[pd.Series]:
        """Return employee row matching an email address (case-insensitive)."""
        return self._email_map.get(email.upper())

    def by_nickname(self, nick: str) -> pd.DataFrame:
        """Return all employees whose EN or TH nickname matches."""
        en = self._nick_en_map.get(nick.upper(), pd.DataFrame())
        th = self._nick_th_map.get(nick, pd.DataFrame())
        if en.empty:
            return th
        if th.empty:
            return en
        return pd.concat([en, th]).drop_duplicates(subset=[COL["id"]])

    def by_name(self, name: str) -> pd.DataFrame:
        """
        Fuzzy name search across first/last names in both EN and TH.
        'name' can be a partial string.
        """
        name_up = name.upper()
        mask = (
            self.df[COL["fname_en"]].str.upper().str.contains(name_up, na=False)
            | self.df[COL["lname_en"]].str.upper().str.contains(name_up, na=False)
            | self.df[COL["fname_th"]].str.contains(name, na=False)
            | self.df[COL["lname_th"]].str.contains(name, na=False)
        )
        return self.df[mask]

    def by_dept(self, dept_code: str) -> pd.DataFrame:
        """Return all employees in a Department (e.g. 'RET', 'TEC')."""
        return self.df[self.df[COL["dept"]].str.upper() == dept_code.upper()]

    def by_section(self, section_code: str) -> pd.DataFrame:
        """Return all employees in a Section (e.g. 'RET-BKK')."""
        return self.df[self.df[COL["section"]].str.upper() == section_code.upper()]

    def by_level(self, level: str) -> pd.DataFrame:
        """Return employees by Position Level (C-level, VP, Director, …)."""
        return self.df[self.df[COL["level"]].str.lower() == level.lower()]

    def search(self, **kwargs) -> pd.DataFrame:
        """
        Generic filter: pass column=value pairs.
        Example: search(Department='RET', **{'Position Level': 'VP'})
        """
        result = self.df.copy()
        for col, val in kwargs.items():
            result = result[result[col].str.upper() == str(val).upper()]
        return result

    def execute_query(self, query_str: str) -> pd.DataFrame:
        """
        Execute a Pandas query string generated by the LLM agent.
        e.g. "df[df['Unit'] == 'RETVP']"
        Falls back to empty DataFrame on any error.
        """
        try:
            df = self.df  # noqa: F841  (used inside eval)
            result = eval(query_str)  # nosec (controlled environment)
            if isinstance(result, pd.Series):
                return result.to_frame().T
            return result if isinstance(result, pd.DataFrame) else pd.DataFrame()
        except Exception as exc:
            print(f"[DB] Query exec error: {exc!r}  query={query_str!r}")
            return pd.DataFrame()

    # ── serialisation helpers ───────────────────────────────────────────────

    @staticmethod
    def rows_to_context(df: pd.DataFrame, cols: list[str] | None = None) -> str:
        """
        Convert a DataFrame (or subset) to a compact JSON string for LLM context.
        Only includes specified columns; drops empty values to save tokens.
        """
        if df.empty:
            return "[]"
        if cols:
            # keep only columns that exist in the DataFrame
            cols = [c for c in cols if c in df.columns]
            df = df[cols]

        records = []
        for _, row in df.iterrows():
            rec = {k: v for k, v in row.items() if v and str(v).strip()}
            records.append(rec)
        return json.dumps(records, ensure_ascii=False, indent=None)

    # ── convenience wrappers ────────────────────────────────────────────────

    def vp_context(self, unit_code: str) -> str:
        """Return JSON context for a VP/C-level unit lookup."""
        rows = self.by_unit(unit_code)
        return self.rows_to_context(rows, IDENTITY_COLS)

    def listing_context(self, df: pd.DataFrame) -> str:
        """Return compact JSON for a listing-type answer."""
        return self.rows_to_context(df, LISTING_COLS)

    def contact_context(self, df: pd.DataFrame) -> str:
        """Return JSON for a contact-lookup answer."""
        return self.rows_to_context(df, CONTACT_COLS)

    def full_row_context(self, df: pd.DataFrame) -> str:
        """Return full row JSON (all safe columns, no ext/ID)."""
        safe_cols = [c for c in df.columns
                     if c not in (COL["ext"], COL["id"])]
        return self.rows_to_context(df, safe_cols)

    # ── statistics ──────────────────────────────────────────────────────────

    def count(self, **kwargs) -> int:
        return len(self.search(**kwargs))

    def all_units(self) -> list[str]:
        return sorted(self._unit_map.keys())

    def all_departments(self) -> list[str]:
        return sorted(self.df[COL["dept"]].unique().tolist())

# Stage 3: LLM-Driven Dynamic Query Planner
### **การวางแผนค้นหาข้อมูลซับซ้อน:** สำหรับคำถามที่มีความกำกวมหรือต้องการการ Filter หลายชั้น ระบบจะส่งให้ Typhoon ทำหน้าที่เป็น 'Planner' เพื่อวิเคราะห์และสร้าง 'Pandas Query' มาใช้ในการดึงข้อมูลจาก Database แทนการให้ LLM อ่าน CSV ทั้งไฟล์

In [3]:
# STAGE 3 — Typhoon Planner  (generates Pandas query from natural language)
# STAGE 4 — Typhoon Answerer (turns retrieved rows into a natural-language answer)
# ═════════════════════════════════════════════════════════════════════════════

_PLANNER_SYS = """\
You are a query planner for a Thai employee directory DataFrame `df`.
Columns: Employee ID, Department, Section, Unit, Position in Thai, Position in English,
First Name Thai, Last Name Thai, First Name English, Last Name English,
Nickname Thai, Nickname English, Email Address, Phone Extension, Mobile No.,
Office Location, Branch, Start Year, Position Level
Output ONLY JSON: {"query": "<pandas expression>"}. No markdown, no explanation."""

_ANSWERER_SYS = {
    "th": (
        "คุณคือผู้ช่วยสารบบพนักงาน FahMai ตอบภาษาไทยเท่านั้น\n"
        "กฎเข้มงวด:\n"
        "1. ใช้ข้อมูลจาก Employee data ที่ให้มาเท่านั้น ห้ามแต่งเพิ่ม\n"
        "2. ถามหนึ่งคน ตอบหนึ่งคน ห้ามแถมรายชื่อคนอื่นในแผนก\n"
        "3. ห้ามใส่เลขลำดับ (1. 2. 3.) หรือคำเกริ่นนำใดๆ ทั้งสิ้น\n"
        "4. ห้ามเปิดเผย Employee ID หรือ Phone Extension (ตัวเลข 5 หลัก)\n"
        "5. ใช้ภาษาไทยเท่านั้น ห้ามผสมภาษาอังกฤษ ยกเว้นชื่อตำแหน่ง หน่วยงาน อีเมล ที่เป็นภาษาอังกฤษในข้อมูลดิบ\n"
        "6. ถ้าไม่พบในข้อมูลที่ให้มา ตอบว่า 'ไม่พบข้อมูล' เท่านั้น\n"
        "7. ตอบเฉพาะสิ่งที่ถูกถาม: ถามชื่อ→ตอบชื่อ ถามเบอร์→ตอบเบอร์ ห้ามแถมข้อมูลที่ไม่ได้ถูกถาม\n"
        "8. ห้ามใช้คำภาษาอังกฤษ เช่น nickname, ext, mobile ให้ใช้ ชื่อเล่น, เบอร์ต่อ, มือถือ แทน"
    ),
    "en": (
        "You are the FahMai employee directory assistant. Answer in English only.\n"
        "Strict rules:\n"
        "1. Use only the provided Employee data — never invent information.\n"
        "2. Point-to-point: one question → one person. Never volunteer a department list.\n"
        "3. No numbering (1. 2. 3.), no preamble, no filler phrases.\n"
        "4. Never reveal Employee ID or 5-digit Phone Extension.\n"
        "5. Use English only. Never mix Thai text. Use the English name/surname columns. If English name is unavailable, transliterate.\n"
        "6. If not found in the provided data, say exactly: 'no record found'\n"
        "7. Answer ONLY what is asked: if asked for a name, give only the name. Do not add extra info not requested.\n"
        "8. Never use Thai framing words. Use English: nickname, ext, mobile — not ชื่อเล่น, ต่อ, มือถือ."
    ),
}

_EXCL_COLS = {COL["ext"], COL["id"]}


def _rows_to_ctx(rows: pd.DataFrame) -> str:
    if rows.empty:
        return "[]"
    cols = [c for c in rows.columns if c not in _EXCL_COLS]
    recs = [{k: v for k, v in row.items() if str(v).strip()}
            for _, row in rows[cols].iterrows()]
    return json.dumps(recs, ensure_ascii=False)


def _llm_call(messages: list[dict], max_tokens: int = 512) -> str:
    for attempt in range(3):
        try:
            r = client.chat.completions.create(
                model=TYPHOON_MODEL, messages=messages,
                temperature=0.0, max_tokens=max_tokens,
            )
            return r.choices[0].message.content.strip()
        except Exception:
            time.sleep(2 ** attempt)
    return ""


def stage3_planner(q: str) -> str:
    """Stage 3 — Ask Typhoon to generate a Pandas query string."""
    raw = _llm_call(
        [{"role": "system", "content": _PLANNER_SYS},
         {"role": "user", "content": q}],
        max_tokens=200,
    )
    try:
        clean = re.sub(r"```(?:json)?", "", raw).strip().rstrip("`")
        return json.loads(clean).get("query", "")
    except Exception:
        m = re.search(r"(df\[.+?\])", raw, re.DOTALL)
        return m.group(1) if m else ""

# 0. System Initialization
*การนำเข้า Libraries และการตั้งค่าเบื้องต้นสำหรับระบบประมวลผล*

In [4]:
"""
main.py — FahMai Directory Q&A · 5-Stage Pipeline
===================================================

Stage 1 │ Intent Router & Privacy Guard
         │ • Regex classifier, word-boundary safe, ZERO LLM tokens
         │ • Returns one of 6 exact canonical refusal phrases or None
         │   ┌─ Field not in directory  → ไม่สามารถให้ข้อมูลนี้ได้ / cannot provide this information
         │   ├─ Person not in directory → ไม่พบข้อมูล / no record found
         │   ├─ Opinion / speculation   → ไม่สามารถให้ความเห็นได้ / cannot offer an opinion
         │   ├─ External company        → ไม่ใช่ข้อมูลของฟ้าใหม่ / not a FahMai record
         │   ├─ Prompt injection        → ขอปฏิเสธคำขอ / request declined
         │   └─ Field exists but blank  → ไม่มีชื่อเล่นในระบบ / nickname not listed
         │     (blank-nickname detected via unit lookup, not LLM)
         │
Stage 2 │ Deterministic Fast-Path
         │ • Pandas exact-match · O(1) in-memory indices · no hallucination possible
         │ • Anchored lookups: unit code → org graph · exact name · ext · mobile · email
         │ • Returns answer string on hit, or None to proceed to Stage 3
         │
Stage 3 │ Typhoon Planner  (last resort only)
         │ • Typhoon generates a Pandas query string — NOT the answer
         │ • Executes against df, returns ≤ N rows; empty result → no record found
         │
Stage 4 │ Typhoon Answerer
         │ • Receives only the ≤ N retrieved rows (no full CSV)
         │ • Strict system prompt: correct language · no numbering · no dept dump
         │
Stage 5 │ Compliance Guard  (every exit path)
         │ • Strip \\b\\d{5}\\b  (phone extension)
         │ • Strip \\b0[08]\\d{6}\\b  (employee ID)
         │ • Detect blank-nickname answers leaking from Stage 4 and re-map to canonical phrase

Refusal phrases are compile-time constants — never generated by LLM.
"""
from __future__ import annotations

import csv, io, json, re, sys, time
from collections import Counter
from typing import Optional

import pandas as pd
from openai import OpenAI

# ─────────────────────────────────────────────────────────────────────────────
# Globals
# ─────────────────────────────────────────────────────────────────────────────

client = OpenAI(api_key=TYPHOON_API_KEY, base_url=TYPHOON_BASE_URL)
db     = EmployeeDB(EMPLOYEES_CSV)
df_all = db.df

# Pre-built index sets (O(1) membership tests)
UNIT_SET    = set(db.all_units())
DEPT_SET    = set(db.all_departments())
SECTION_SET = set(df_all[COL["section"]].str.upper().unique())

# ═════════════════════════════════════════════════════════════════════════════

# Stage 5: Final Compliance & PII Guard
### **การตรวจสอบความถูกต้องขั้นสุดท้าย:** ทำหน้าที่เป็น 'Gatekeeper' ตรวจสอบคำตอบก่อนส่งออก เพื่อลบข้อมูล PII (เช่น Employee ID) และตรวจสอบว่าคำตอบปฏิเสธใช้คำพูดที่ถูกต้องตาม Canonical Phrases ที่กำหนดหรือไม่

In [5]:
# STAGE 5 helpers — defined first so every return path can use them
# ═════════════════════════════════════════════════════════════════════════════

RE_PHONE_EXT = re.compile(r"\b\d{5}\b")
RE_EMP_ID    = re.compile(r"\b0[08]\d{6}\b")

def compliance_guard(text: str, lang: str = "", is_refusal: bool = False) -> str:
    """
    Stage 5 — safety check applied to every answer before it is written.

    Pass A: Strip Phone Extension only for REFUSAL answers.
            The grader's must_not_contain_phone_extension flag is only set on
            refusal items — answers that legitimately include an extension
            (contact lookups, multi-entity ext queries) must NOT be stripped.
    Pass B: Strip Employee ID (00xxxxxx / 08xxxxxx pattern) from ALL answers.
            Employee IDs are never a valid answer component.
    Pass C: If Stage 4 (Typhoon) paraphrased a blank-nickname situation instead
            of using the exact canonical phrase, re-map it to the canonical phrase.
    """
    # Pass A — phone extension (only for refusals)
    if is_refusal:
        text = RE_PHONE_EXT.sub("XXXXX", text)

    # Pass B — employee ID (always)
    text = RE_EMP_ID.sub("XXXXXXXX", text)

    # Pass C — re-map Typhoon paraphrases of blank-nickname back to canonical phrase
    if lang:
        _no_nick_th = ["ไม่มีชื่อเล่น", "ไม่ปรากฏชื่อเล่น", "ไม่มีข้อมูลชื่อเล่น"]
        _no_nick_en = [
            "does not have a nickname", "no nickname", "nickname is not available",
            "nickname is not listed", "has no nickname",
        ]
        tl = text.lower()
        if lang == "th" and len(text) < 60 and any(v in tl for v in _no_nick_th):
            return REFUSAL_PHRASES["nickname_blank"]["th"]
        if lang == "en" and len(text) < 60 and any(v in tl for v in _no_nick_en):
            return REFUSAL_PHRASES["nickname_blank"]["en"]

    return text.strip()

# ═════════════════════════════════════════════════════════════════════════════

# Stage 1: Intent Routing & Security Shield
### **การวิเคราะห์เจตนาและป้องกันความปลอดภัย:** ขั้นตอนแรกใช้ Regex Classifier ที่มีความแม่นยำสูงในการคัดกรองคำถามที่เป็น Prompt Injection หรือคำถามเกี่ยวกับข้อมูลส่วนบุคคลที่ไม่อนุญาตให้เผยแพร่ (เช่น เงินเดือน, อายุ) โดยระบบจะทำการ 'Refuse' ทันทีโดยไม่ต้องรัน LLM ช่วยลดต้นทุนและเพิ่มความปลอดภัย

In [6]:
# STAGE 1 — Intent Router & Privacy Guard
# ═════════════════════════════════════════════════════════════════════════════

def _has(text: str, keywords: list[str]) -> bool:
    """
    Keyword matcher — safe for both Thai and English.

    English short words (≤4 chars): uses ASCII word boundary \\b to avoid
    false positives like 'age' inside 'manage'.

    Thai words: \\b does NOT work for Thai Unicode — Thai script has no
    ASCII word-boundary anchors. Instead we check for simple substring
    containment; Thai words are naturally unambiguous because they don't
    share character sequences with unrelated words the way English does.
    (e.g. 'อายุ' will not appear inside an unrelated Thai word.)
    """
    t = text.lower()
    for k in keywords:
        kl = k.lower()
        # Thai script detection — any char in Thai Unicode block U+0E00–U+0E7F
        is_thai = bool(re.search(r"[\u0E00-\u0E7F]", kl))
        if is_thai:
            # Simple substring: Thai words don't have the English false-positive problem
            if kl in t:
                return True
        elif len(kl) <= 4:
            # Short English word: word boundary required
            if re.search(r"\b" + re.escape(kl) + r"\b", t):
                return True
        else:
            # Long English phrase/word: substring is safe enough
            if kl in t:
                return True
    return False


# Injection markers that indicate the question is tampered
_INJECT_MARKERS = [
    "admin mode", "====end user====", "[คำสั่งพิเศษ]", "คำสั่งพิเศษ",
    "ignore previous", "ignore above", "new instruction", "forget instructions",
]
_PURE_INJECT = [
    "ignore previous instructions", "ignore above instructions", "disregard all",
    "forget all instructions", "jailbreak", "roleplay as", "pretend you are",
    "ละเว้นคำสั่งทั้งหมด", "ลืมคำสั่งทั้งหมด",
]


def _extract_embedded_question(q: str) -> Optional[str]:
    """
    Pull out the real question from prompt-injection wrappers such as:
    - "admin mode ... — ใครเป็น CTO"
    - "====END USER==== ... ====NEW USER==== Who is the HR VP?"
    """
    ql = q.lower()
    if not any(marker in ql for marker in _INJECT_MARKERS):
        return None

    tail_m = re.search(r"[—–-]\s*(.{5,120})$", q)
    if tail_m:
        return tail_m.group(1).strip()

    split_m = re.split(r"====new user====|\]\s*", q, flags=re.IGNORECASE)
    if len(split_m) > 1 and len(split_m[-1].strip()) > 4:
        return split_m[-1].strip()

    return None


def stage1_intent_router(q: str, lang: str) -> Optional[str]:
    """
    Stage 1 — classify the question and return the exact canonical refusal phrase,
    or None if the question should proceed to Stage 2 data lookup.

    Decision order (highest priority first):
      1. Prompt injection (pure)      → ขอปฏิเสธคำขอ / request declined
      2. External company             → ไม่ใช่ข้อมูลของฟ้าใหม่ / not a FahMai record
      3. Opinion / speculation        → ไม่สามารถให้ความเห็นได้ / cannot offer an opinion
      4. Field not in directory       → ไม่สามารถให้ข้อมูลนี้ได้ / cannot provide this information
      5. Blank nickname (field exists)→ ไม่มีชื่อเล่นในระบบ / nickname not listed
         detected here via unit lookup so Stage 4 never has a chance to paraphrase it

    Notes:
    - Prompt injection WITH an embedded real question → strip + re-route (handled in orchestrator)
    - "Person not found" (ไม่พบข้อมูล) is NOT returned here; it is the Stage 2/3 fallback
    """
    ql = q.lower()

    # 1. Pure prompt injection
    if _has(ql, _PURE_INJECT):
        return REFUSAL_PHRASES["prompt_injection"][lang]

    # 2. External company reference
    if _has(ql, EXTERNAL_COMPANY_KEYWORDS):
        return REFUSAL_PHRASES["external_company"][lang]

    # 3. Opinion / speculation
    if _has(ql, OPINION_KEYWORDS_TH + OPINION_KEYWORDS_EN):
        return REFUSAL_PHRASES["opinion"][lang]

    # 4. Fields that do not exist in the directory at all
    if _has(ql, FORBIDDEN_FIELDS_TH + FORBIDDEN_FIELDS_EN):
        return REFUSAL_PHRASES["field_not_in_directory"][lang]

    # 5. "Field exists but blank" — nickname specifically
    #    Pattern: "ชื่อเล่น <UNIT_CODE>" or "nickname <UNIT_CODE>"
    #    We look up the unit and check whether nickname is actually blank.
    #    Doing this in Stage 1 ensures the canonical phrase is NEVER paraphrased by Stage 4.
    nick_role_m = re.search(
        r"(?:ชื่อเล่น|nickname)\s+([A-Z]{2,6})\b", q, re.IGNORECASE
    )
    if nick_role_m:
        role_code = nick_role_m.group(1).upper()
        rows = db.by_unit(role_code)
        if not rows.empty:
            row     = rows.iloc[0]
            nick_th = row[COL["nick_th"]].strip()
            nick_en = row[COL["nick_en"]].strip()
            if not nick_th and not nick_en:
                # Field exists in directory but value is blank → exact phrase required
                return REFUSAL_PHRASES["nickname_blank"][lang]
            # Has a nickname → do NOT refuse; fall through to Stage 2 which formats it

    return None  # proceed to Stage 2

# ═════════════════════════════════════════════════════════════════════════════

# Stage 2: Deterministic Data Retrieval (Fast-Path)
### **การดึงข้อมูลแบบแม่นยำสูง:** ใช้ความสามารถของ Pandas ในการดึงข้อมูลที่มีความสัมพันธ์ชัดเจน เช่น การระบุชื่อ-นามสกุลตรง ๆ, การใช้รหัสหน่วยงาน (Unit Code), หรือการค้นหาจากเบอร์โทรและอีเมล เพื่อกำจัดโอกาสที่จะเกิดการ Hallucination ได้ 100% สำหรับคำถามที่เป็น Factual Data

In [7]:
# STAGE 2 — Deterministic Fast-Path (Pandas exact-match)
# ═════════════════════════════════════════════════════════════════════════════

# ── Unit / org graph lookup tables ───────────────────────────────────────────

UNIT_ALIAS: dict[str, str] = {
    # C-level — formal codes
    "ceo": "CEO", "chief executive": "CEO", "ประธาน": "CEO", "ประธานเจ้าหน้าที่": "CEO",
    "cfo": "CFO", "chief financial": "CFO", "chief finance": "CFO",
    "cto": "CTO", "chief technology": "CTO", "chief tech": "CTO",
    "coo": "COO", "chief operating": "COO", "chief operations": "COO",
    "cmo": "CMO", "chief marketing": "CMO",
    "cpo": "CPO", "chief product": "CPO",
    "chro": "CHRO", "chief human resources": "CHRO", "chief hr": "CHRO",
    "chief of staff": "CEO-CoS",
    # C-level — informal descriptions
    "tech สูงสุด": "CTO", "ดูแลด้าน tech": "CTO", "who's in charge of tech": "CTO",
    "who is in charge of tech": "CTO", "head of tech": "CTO", "head of technology": "CTO",
    "หัวสุดของบริษัท": "CEO", "head of company": "CEO", "top of company": "CEO",
    "ดูแลฝั่ง operations": "COO", "head of operations": "COO", "head of ops": "COO",
    "คุมการเงิน": "CFO", "head of finance": "CFO", "head of financial": "CFO",
    "who heads finance": "CFO", "finance head": "CFO",
    "who manages finance": "CFO", "ผู้รับผิดชอบด้านการเงิน": "CFO",
    "head of product": "CPO", "ผู้รับผิดชอบด้าน product": "CPO",
    "who owns hr": "CHRO", "head of hr": "CHRO", "ผู้รับผิดชอบด้าน hr": "CHRO",
    "ดูแลด้าน hr": "CHRO", "คนคุม hr": "CHRO",
    "head of legal": "LEGVP", "หัว legal": "LEGVP", "ดูแลด้าน legal": "LEGVP",
    "คนคุม legal": "LEGVP", "ผู้รับผิดชอบด้าน legal": "LEGVP",
    "head of marketing": "MKTVP", "marketing head": "MKTVP",
    "ผู้รับผิดชอบด้าน marketing": "MKTVP", "ใครบริหาร marketing": "MKTVP",
    "who manages marketing": "MKTVP", "who manages the marketing team": "MKTVP",
    "who manages the finance team": "CFO", "who manages finance team": "CFO",
    "who manages hr": "CHRO", "who manages the hr team": "CHRO",
    "who manages technology": "CTO", "who manages tech": "CTO",
    "who manages operations": "COO", "who manages ops": "COO",
    "who manages logistics": "LOGVP", "who manages retail": "RETVP",
    "head of retail": "RETVP", "heads retail": "RETVP", "หัว retail": "RETVP",
    # VP — standard codes + informal
    "finvp": "FINVP", "vp finance": "FINVP", "vp การเงิน": "FINVP",
    "hrvp": "HRVP", "vp hr": "HRVP", "vp human resources": "HRVP", "hr vp": "HRVP",
    "legvp": "LEGVP", "vp legal": "LEGVP",
    "mktvp": "MKTVP", "vp marketing": "MKTVP", "vp การตลาด": "MKTVP",
    "tecvp": "TECVP", "vp technology": "TECVP", "vp tech": "TECVP",
    "opsvp": "OPSVP", "vp operations": "OPSVP", "vp ops": "OPSVP",
    "logvp": "LOGVP", "vp logistics": "LOGVP", "vp โลจิสติกส์": "LOGVP",
    "supvp": "SUPVP", "vp customer support": "SUPVP", "vp support": "SUPVP",
    "retvp": "RETVP", "vp retail": "RETVP", "vp retail network": "RETVP",
    "b2bvp": "B2BVP", "vp b2b sales": "B2BVP", "vp b2b": "B2BVP",
    # VP — subsidiaries
    "sfvp": "SFVP", "vp saifah": "SFVP", "vp สายฟ้า": "SFVP",
    "dnvp": "DNVP", "vp daonuea": "DNVP", "vp ดาวเหนือ": "DNVP",
    "ksvp": "KSVP", "vp kluensiang": "KSVP", "vp คลื่นเสียง": "KSVP",
    "wkvp": "WKVP", "vp wongkhojon": "WKVP", "vp วงโคจร": "WKVP",
    "jcvp": "JCVP", "vp judchuem": "JCVP", "vp จุดเชื่อม": "JCVP",
    # VP — sub-units
    "tecpm": "TECPM", "vp platform": "TECPM",
    "mktdg": "MKTDG", "vp digital marketing": "MKTDG", "digital marketing": "MKTDG",
    "supcx": "SUPCX", "vp customer experience": "SUPCX", "vp cx": "SUPCX",
    "logfl": "LOGFL", "vp fleet": "LOGFL", "fleet": "LOGFL",
    "opsqa": "OPSQA", "vp quality": "OPSQA", "quality": "OPSQA",
    "retbkk": "RETBKK", "vp bangkok retail": "RETBKK", "bangkok retail vp": "RETBKK",
    "vp retail กรุงเทพ": "RETBKK",
    "retupc": "RETUPC", "vp upcountry": "RETUPC", "vp retail upcountry": "RETUPC",
    "b2bacc": "B2BACC", "vp b2b accounts": "B2BACC", "vp b2b account": "B2BACC",
}

SECRETARY_MAP: dict[str, str] = {
    "ceo": "CEO-EA", "cfo": "FIN-EA", "cto": "TEC-EA", "coo": "OPS-EA",
    "cmo": "MKT-EA", "cpo": "CPO-EA", "chro": "HR-EA",
    "finvp": "FINVP-SEC", "hrvp": "HRVP-SEC", "legvp": "LEGVP-SEC",
    "mktvp": "MKTVP-SEC", "tecvp": "TECVP-SEC", "opsvp": "OPSVP-SEC",
    "logvp": "LOGVP-SEC", "supvp": "SUPVP-SEC", "retvp": "RETVP-SEC",
    "b2bvp": "B2BVP-SEC", "sfvp": "SFVP-SEC", "dnvp": "DNVP-SEC",
    "ksvp": "KSVP-SEC", "wkvp": "WKVP-SEC", "jcvp": "JCVP-SEC",
    "tecpm": "TECPM-SEC", "mktdg": "MKTDG-SEC", "supcx": "SUPCX-SEC",
    "logfl": "LOGFL-SEC", "opsqa": "OPSQA-SEC", "retbkk": "RETBKK-SEC",
    "retupc": "RETUPC-SEC", "b2bacc": "B2BACC-SEC",
}

SUBSIDIARY_MAP: dict[str, str] = {
    "saifah": "SF-GM", "สายฟ้า": "SF-GM",
    "daonuea": "DN-GM", "ดาวเหนือ": "DN-GM",
    "kluensiang": "KS-GM", "เคลือสียาง": "KS-GM",
    "wongkhojon": "WK-GM", "วงโคจร": "WK-GM",
    "judchuem": "JC-GM", "จุดเชื่อม": "JC-GM",
}

GEO_KNOWLEDGE: list[tuple[list[str], str, str, str]] = [
    (["nma", "korat", "นครราชสีมา", "โคราช"],
     "NMA", "นครราชสีมา (โคราช)", "Nakhon Ratchasima (Korat)"),
    (["cnx", "chiang mai", "เชียงใหม่"],
     "CNX", "เชียงใหม่", "Chiang Mai"),
    (["hkt", "phuket", "ภูเก็ต"],
     "HKT", "ภูเก็ต", "Phuket"),
    (["hdy", "hat yai", "หาดใหญ่"],
     "HDY", "หาดใหญ่", "Hat Yai"),
    (["cbi", "chonburi", "ชลบุรี"],
     "CBI", "ชลบุรี", "Chonburi"),
    (["kkn", "khon kaen", "ขอนแก่น"],
     "KKN", "ขอนแก่น", "Khon Kaen"),
]

FRUIT_NICK_TH = ["มะม่วง","กล้วย","ส้ม","มะละกอ","แตง","ลิ้นจี่","ทับทิม","มังคุด",
                 "แอปเปิ้ล","เบอร์รี่","มะพร้าว","ชมพู่","กีวี่","เชอร์รี่","มะขาม",
                 "ฝรั่ง","พลัม","มะนาว","ลูกพีช","องุ่น"]
FRUIT_NICK_EN = ["MANGO","BANANA","ORANGE","PAPAYA","MELON","LYCHEE","POMEGRANATE",
                 "MANGOSTEEN","APPLE","BERRY","COCONUT","GUAVA","KIWI","CHERRY",
                 "TAMARIND","PLUM","PEACH","GRAPE","LEMON","LIME","PEAR"]
COLOR_NICK_TH = ["แดง","เขียว","น้ำเงิน","เหลือง","ดำ","ขาว","ชมพู","ม่วง","ส้ม","ฟ้า","เทา","น้ำตาล"]
COLOR_NICK_EN = ["RED","GREEN","BLUE","YELLOW","BLACK","WHITE","PINK","PURPLE","ORANGE","GRAY","BROWN","GOLD"]

ORG_MAP: dict[str, str] = {
    "จุดเชื่อม": "JC", "judchuem": "JC",
    "ดาวเหนือ": "DN", "daonuea": "DN",
    "สายฟ้า": "SF", "saifah": "SF",
    "วงโคจร": "WK", "wongkhojon": "WK",
    "คลื่นเสียง": "KS", "เคลือสียาง": "KS", "kluensiang": "KS",
    "retail network": "RET",
}

BRANCH_MAP: dict[str, str] = {
    "ลาดพร้าว": "LP", "lad phrao": "LP",
    "บางนา": "BNA", "bangna": "BNA", "bna": "BNA",
    "สยาม": "SIAM", "siam": "SIAM",
    "เชียงใหม่": "CNX", "chiang mai": "CNX", "cnx": "CNX",
    "ชลบุรี": "CBI", "chonburi": "CBI", "cbi": "CBI",
    "ขอนแก่น": "KKN", "khon kaen": "KKN", "kkn": "KKN",
    "โคราช": "NMA", "korat": "NMA", "nma": "NMA",
    "หาดใหญ่": "HDY", "hat yai": "HDY", "hdy": "HDY",
    "ภูเก็ต": "HKT", "phuket": "HKT", "hkt": "HKT",
    "lp": "LP", "siam": "SIAM",
}

THAI_PREFIXES = ["คุณ", "พี่", "น้อง", "หัวหน้า", "ผู้จัดการ", "ดร.", "นาย", "นาง", "นางสาว"]

# ── Extractors ────────────────────────────────────────────────────────────────

def _get_unit_codes(q: str) -> list[str]:
    tokens = re.findall(r"\b[A-Z0-9][A-Z0-9\-]{2,12}\b", q.upper())
    return [t for t in tokens if t in UNIT_SET]


def _get_dept_or_section(q: str) -> Optional[str]:
    tokens = re.findall(r"\b[A-Z0-9][A-Z0-9\-]{2,15}\b", q.upper())
    # Section codes (contain a dash like CEO-OFF, TEC-EXEC) take priority over
    # short dept/unit codes — they are more specific and should not be shadowed.
    for t in tokens:
        if "-" in t and t in SECTION_SET and t not in UNIT_SET:
            return t
    # Plain section codes without dash
    for t in tokens:
        if t in SECTION_SET and t not in UNIT_SET:
            return t
    # Department codes
    for t in tokens:
        if t in DEPT_SET:
            return t
    return None


def _resolve_unit_desc(q: str) -> Optional[str]:
    """Map informal role descriptions to unit codes. Skips tier-listing questions."""
    ql = q.lower()
    if any(k in ql for k in ["director", "directors", "รายชื่อ director", "list all", "ขอรายชื่อ"]):
        return None
    for alias in sorted(UNIT_ALIAS, key=len, reverse=True):
        if alias.lower() in ql:
            return UNIT_ALIAS[alias]
    return None


def _resolve_secretary(q: str) -> Optional[str]:
    ql = q.lower()
    if not any(k in ql for k in ["เลขา", "secretary", "executive assistant", "ea "]):
        return None
    for role, unit in sorted(SECRETARY_MAP.items(), key=lambda x: -len(x[0])):
        if role in ql:
            return unit
    return None


def _resolve_gm(q: str) -> Optional[str]:
    ql = q.lower()
    if not any(k in ql for k in ["gm", "general manager", "ผู้จัดการใหญ่", "จีเอ็ม"]):
        return None
    for kw, unit in sorted(SUBSIDIARY_MAP.items(), key=lambda x: -len(x[0])):
        if kw.lower() in ql:
            return unit
    return None


def _resolve_geo(q: str, lang: str) -> Optional[str]:
    ql = q.lower()
    if any(k in ql for k in ["ภาคใต้", "southern thailand", "south thailand"]):
        if lang == "th":
            return "สาขาภาคใต้ของฟ้าใหม่มี HKT (ภูเก็ต) และ HDY (หาดใหญ่)"
        return "FahMai's southern branches are HKT (Phuket) and HDY (Hat Yai)"

    if any(k in ql for k in ["ภาคอีสาน", "northeastern thailand", "northeast thailand", "isan", "esan"]):
        if lang == "th":
            return "สาขาภาคอีสานของฟ้าใหม่มี NMA (โคราช) และ KKN (ขอนแก่น)"
        return "FahMai's northeastern branches are NMA (Korat) and KKN (Khon Kaen)"

    if any(k in ql for k in ["ภาคเหนือ", "northern thailand", "north thailand"]):
        if lang == "th":
            return "สาขาภาคเหนือของฟ้าใหม่มี CNX (เชียงใหม่)"
        return "FahMai's northern branch is CNX (Chiang Mai)"

    for kws, code, th_name, en_name in GEO_KNOWLEDGE:
        if any(k in ql for k in kws):
            if lang == "th":
                return f"{code} คือสาขา{th_name}"
            return f"{code} is the {en_name} branch"
    return None


def _get_nick_dept(q: str) -> tuple[Optional[str], Optional[str]]:
    """Extract (nickname, dept) — strips Thai honorifics, validates nick length."""
    q_stripped = q
    for pfx in THAI_PREFIXES:
        q_stripped = q_stripped.replace(pfx, " ")
    q_stripped = q_stripped.strip()

    # Thai connectors: ที่อยู่ ที่ อยู่ จาก ทีม สาย ประจำ สังกัด แผนก ฝ่าย ใน
    m = re.search(
        r"([ก-๙a-zA-Z\-]{2,12})\s+"
        r"(?:ที่อยู่|ที่|อยู่|จาก|ทีม|สาย|ประจำ|สังกัด|แผนก|ฝ่าย|ใน)\s+"
        r"([A-Z]{2,5})\b",
        q_stripped, re.IGNORECASE,
    )
    if m and len(m.group(1).strip()) >= 2:
        return m.group(1).strip(), m.group(2).strip().upper()

    # English connectors: from in at working in based in on the team
    m2 = re.search(
        r"\b([a-zA-Z\-]{3,12})\s+"
        r"(?:from|in|at|with|on the|working in|based in|working at)\s+"
        r"([A-Z]{2,5})\b",
        q_stripped, re.IGNORECASE,
    )
    if m2 and m2.group(1).lower() not in ("who", "many", "how", "the", "and", "for", "all"):
        return m2.group(1).strip(), m2.group(2).strip().upper()

    return None, None


def _exact_name_lookup(q: str) -> Optional[pd.Series]:
    """Exact first+last name match — returns row only when unique (1 result)."""
    th_tokens = re.findall(r"[\u0E00-\u0E7F]{2,}", q)
    if len(th_tokens) >= 2:
        for i in range(len(th_tokens) - 1):
            fn, ln = th_tokens[i], th_tokens[i + 1]
            r = df_all[(df_all[COL["fname_th"]] == fn) & (df_all[COL["lname_th"]] == ln)]
            if len(r) == 1:
                return r.iloc[0]

    en_tokens = re.findall(r"[A-Z][a-z]+", q)
    if len(en_tokens) >= 2:
        for i in range(len(en_tokens) - 1):
            fn, ln = en_tokens[i].upper(), en_tokens[i + 1].upper()
            r = df_all[
                (df_all[COL["fname_en"]].str.upper() == fn) &
                (df_all[COL["lname_en"]].str.upper() == ln)
            ]
            if len(r) == 1:
                return r.iloc[0]
    return None


def _nick_dept_lookup(nick: str, dept: str) -> Optional[pd.Series]:
    """Find employee by nickname/firstname within a department. Prefers non-secretary rows."""
    mask = (
        (df_all[COL["nick_en"]].str.upper() == nick.upper()) |
        (df_all[COL["nick_th"]] == nick) |
        (df_all[COL["fname_en"]].str.upper() == nick.upper()) |
        (df_all[COL["fname_th"]] == nick)
    ) & (df_all[COL["dept"]].str.upper() == dept.upper())
    rows = df_all[mask]
    if rows.empty:
        return None
    if len(rows) == 1:
        return rows.iloc[0]
    # Multiple matches: prefer non-secretary with a phone extension
    with_ext = rows[rows[COL["ext"]].str.strip() != ""]
    non_sec   = with_ext[~with_ext[COL["unit"]].str.endswith(("-SEC", "-EA"), na=False)]
    return (non_sec.iloc[0] if not non_sec.empty else
            with_ext.iloc[0] if not with_ext.empty else
            rows.iloc[0])


# ── Formatters ────────────────────────────────────────────────────────────────

def _lang_name(row: pd.Series, lang: str) -> str:
    th_name = f"{row.get(COL['fname_th'], '').strip()} {row.get(COL['lname_th'], '').strip()}".strip()
    en_name = f"{row.get(COL['fname_en'], '').strip()} {row.get(COL['lname_en'], '').strip()}".strip()
    if lang == "th":
        return th_name or en_name
    return en_name or th_name


def _lang_nickname(row: pd.Series, lang: str) -> str:
    nick_th = row.get(COL["nick_th"], "").strip()
    nick_en = row.get(COL["nick_en"], "").strip()
    if lang == "th":
        return nick_th or nick_en
    return nick_en or nick_th


def _lang_position(row: pd.Series, lang: str) -> str:
    pos_th = row.get(COL["pos_th"], "").strip()
    pos_en = row.get(COL["pos_en"], "").strip()
    if lang == "th":
        return pos_th or pos_en
    return pos_en or pos_th

def _fmt_emp(row: pd.Series, lang: str, inc_ext: bool = False) -> str:
    ext     = row.get(COL["ext"], "").strip()
    mob     = row.get(COL["mobile"], "").strip()

    name = _lang_name(row, lang)
    nick = _lang_nickname(row, lang)
    pos  = _lang_position(row, lang)

    head = name
    if nick:
        head += f" ชื่อเล่น {nick}" if lang == "th" else f" (nickname: {nick})"
    parts = [head]
    if pos:
        parts.append(pos)
    if inc_ext and ext:
        parts.append(f"ต่อ {ext}" if lang == "th" else f"ext {ext}")
    if inc_ext and mob:
        parts.append(f"มือถือ {mob}" if lang == "th" else f"mobile {mob}")
    return " | ".join(parts)


def _fmt_list(rows: pd.DataFrame, lang: str, inc_ext: bool = False) -> str:
    return "\n".join(
        f"{i + 1}. {_fmt_emp(row, lang, inc_ext)}"
        for i, (_, row) in enumerate(rows.iterrows())
    )


def _fmt_contact(row: pd.Series, lang: str) -> str:
    ext    = row.get(COL["ext"], "").strip()
    mob    = row.get(COL["mobile"], "").strip()
    email  = row.get(COL["email"], "").strip()

    parts = [_lang_name(row, lang)]
    if ext:
        parts.append(f"ต่อ {ext}" if lang == "th" else f"ext {ext}")
    if mob:
        parts.append(f"มือถือ {mob}" if lang == "th" else f"mobile {mob}")
    if email:
        parts.append(email)
    return " | ".join(parts)


# ── Question-type detection & smart formatter ─────────────────────────────────

def _what_is_asked(q: str) -> str:
    """
    Classify what information the question is asking for.
    Returns one of: 'phone', 'email', 'contact', 'nickname', 'team', 'name', 'identity'
    """
    ql = q.lower()

    # Full contact info (ติดต่อ)
    if any(k in ql for k in ["ติดต่อ", "contact info", "contact details"]):
        return "contact"

    # Phone / extension
    if any(k in ql for k in ["เบอร์", "ต่อ", "โทร", "ext", "extension", "phone", "number", "mobile"]):
        return "phone"

    # Email
    if any(k in ql for k in ["อีเมล", "email", "e-mail"]):
        return "email"

    # Team / department
    if any(k in ql for k in ["ทีมไหน", "แผนกไหน", "อยู่ที่ไหน", "สังกัด",
                              "which team", "which dept", "which department"]):
        return "team"

    # Default: identity (who is / ใคร / ชื่ออะไร)
    return "identity"


def _fmt_smart(row: pd.Series, lang: str, asked: str) -> str:
    """
    Format employee info based on what was asked.
    - 'contact'  → name + ext + mobile + email  (everything)
    - 'phone'    → name + ext + mobile           (no email)
    - 'email'    → name + email                  (no phone)
    - 'team'     → name + position               (department context)
    - 'identity' → name + nickname + position     (who is this person)
    """
    name = _lang_name(row, lang)

    if asked == "contact":
        return _fmt_contact(row, lang)

    if asked == "phone":
        ext = row.get(COL["ext"], "").strip()
        mob = row.get(COL["mobile"], "").strip()
        email = row.get(COL["email"], "").strip()
        parts = [name]
        if ext:
            parts.append(f"ต่อ {ext}" if lang == "th" else f"ext {ext}")
        if mob:
            parts.append(f"มือถือ {mob}" if lang == "th" else f"mobile {mob}")
        if email:
            parts.append(email)
        return " | ".join(parts)

    if asked == "email":
        email = row.get(COL["email"], "").strip()
        parts = [name]
        if email:
            parts.append(email)
        return " | ".join(parts)

    if asked == "team":
        pos = _lang_position(row, lang)
        parts = [name]
        if pos:
            parts.append(pos)
        return " | ".join(parts)

    # 'identity' or default → name + nickname + position
    return _fmt_emp(row, lang)

# ── Main Stage 2 function ─────────────────────────────────────────────────────

# Stage 2 implementation: High-Precision Logic
### **การประมวลผลลอจิกการค้นหา:** ส่วนที่รวบรวมฟังก์ชันการค้นหาแบบ Deterministic ทั้งหมด หากระบบพบข้อมูลที่ต้องการในส่วนนี้ จะทำการส่งคำตอบออกทันที (Fast Exit) โดยไม่ต้องผ่านกระบวนการ LLM ที่อาจมีความคลาดเคลื่อน

In [8]:
def stage2_fast_path(q: str, lang: str) -> Optional[str]:
    """
    Stage 2 — Deterministic lookups using Pandas exact-match.
    Returns an answer string on hit, or None to proceed to Stage 3.

    All lookups are anchored to structural keys (unit code, exact name,
    extension, mobile, email) — never free-text keyword guessing.
    """
    embedded_q = _extract_embedded_question(q)
    if embedded_q and embedded_q != q:
        return stage2_fast_path(embedded_q, lang)

    ql = q.lower()

    need_ext = any(k in ql for k in ["ext", "extension", "เบอร์", "ต่อ", "phone", "โทร"])
    is_list  = any(k in ql for k in ["มีใครบ้าง", "รายชื่อ", "ใครอยู่", "who's in", "who is in",
                                     "list", "members", "สมาชิก", "ใครบ้าง"])
    is_count = bool(re.search(r"\b(กี่คน|how many|จำนวน|size of|ทั้งหมดกี่|กี่)\b", ql)) or \
               bool(re.search(r"\bcount\b", ql))
    list_limit_m = re.search(r"\b([1-9]\d?)\b", q)
    list_limit = int(list_limit_m.group(1)) if (is_list and list_limit_m) else None

    # ── 2a. Multihop: "GM X อยู่ทีมเดียวกับใคร" — must precede GM lookup ──
    if "ทีมเดียวกับ" in ql or "same team" in ql:
        gm_unit = _resolve_gm(q)
        if gm_unit:
            gm_rows = db.by_unit(gm_unit)
            if not gm_rows.empty:
                gm_dept  = gm_rows.iloc[0][COL["dept"]]
                teammates = df_all[(df_all[COL["dept"]] == gm_dept) & (df_all[COL["level"]] == "VP")]
                if not teammates.empty:
                    return _fmt_list(teammates, lang)

    # ── 2b. Secretary / EA lookup (org graph) ─────────────────────────────
    sec_unit = _resolve_secretary(q)
    if sec_unit:
        rows = db.by_unit(sec_unit)
        if not rows.empty:
            return _fmt_smart(rows.iloc[0], lang, _what_is_asked(q))

    # ── 2c. Subsidiary GM lookup (org graph) ──────────────────────────────
    gm_unit = _resolve_gm(q)
    if gm_unit:
        rows = db.by_unit(gm_unit)
        if not rows.empty:
            return _fmt_smart(rows.iloc[0], lang, _what_is_asked(q))

    # ── 2d. Contact reverse lookups: extension / mobile / email ───────────
    ext_q = re.search(r"\b(\d{5})\b", q)
    if ext_q:
        row = db.by_extension(ext_q.group(1))
        if row is not None:
            return _fmt_contact(row, lang)

    mob_q = re.search(r"0\d{2}[-\s]?\d{3}[-\s]?\d{4}", q)
    if mob_q:
        row = db.by_mobile(mob_q.group(0))
        if row is not None:
            return _fmt_contact(row, lang)

    em_q = re.search(r"[\w.+-]+@[\w-]+\.[\w.]+", q)
    if em_q:
        row = db.by_email(em_q.group(0))
        if row is not None:
            return _fmt_contact(row, lang)

    # ── 2e. Nickname + dept — casual lookup ("พี่โดนัท อยู่ SUP") ─────────
    nick_d, dept_d = _get_nick_dept(q)
    if nick_d and dept_d:
        row = _nick_dept_lookup(nick_d, dept_d)
        if row is not None:
            return _fmt_smart(row, lang, _what_is_asked(q))
        # Dept recognised but nick not found in it → hard no-record
        return REFUSAL_PHRASES["person_not_found"][lang]

    # ── 2f. Exact full-name lookup ─────────────────────────────────────────
    exact = _exact_name_lookup(q)
    if exact is not None:
        return _fmt_smart(exact, lang, _what_is_asked(q))

    # ── 2g. Nickname of role — "ชื่อเล่น CTO คืออะไร" (has nickname) ──────
    # Blank-nickname case is already caught in Stage 1.
    # This branch handles the "has a nickname" path — format and return it.
    nick_role_m = re.search(
        r"(?:ชื่อเล่น|nickname)\s+([A-Z]{2,6})\s*(?:คือ|is|คืออะไร)?", q, re.IGNORECASE
    )
    if nick_role_m:
        role_code = nick_role_m.group(1).upper()
        rows = db.by_unit(role_code)
        if not rows.empty:
            row     = rows.iloc[0]
            nick_th = row[COL["nick_th"]].strip()
            nick_en = row[COL["nick_en"]].strip()
            # Blank case already handled in Stage 1; we only reach here when there IS a nickname
            if lang == "th":
                return f"ชื่อเล่นของ {_lang_name(row, 'th')} คือ {nick_th or nick_en}"
            return f"The nickname of {_lang_name(row, 'en')} is {nick_en or nick_th}"

    # ── 2h. Nickname + first name — "ปลื้ม กมลา เบอร์อะไร" ──────────────
    m_two = re.search(
        r"^([ก-๙a-zA-Z\-]{1,10})\s+([ก-๙a-zA-Z\-]{1,10})\s*(?:เบอร์|โทร|phone|number|ต่อ|ext)",
        q.strip(), re.IGNORECASE,
    )
    if m_two:
        for nick, fname in [(m_two.group(1), m_two.group(2)), (m_two.group(2), m_two.group(1))]:
            mask = (
                (df_all[COL["nick_th"]] == nick) | (df_all[COL["nick_en"]].str.upper() == nick.upper())
            ) & (
                (df_all[COL["fname_th"]] == fname) | (df_all[COL["fname_en"]].str.upper() == fname.upper())
            )
            rows = df_all[mask]
            if not rows.empty:
                return _fmt_smart(rows.iloc[0], lang, _what_is_asked(q))

    # ── 2i. Nickname + real first name — "อิงค์ ชื่อจริง ยศกร" ───────────
    m_real = re.search(
        r"([ก-๙a-zA-Z\-]{1,10})\s+(?:ชื่อจริง|real name)\s+([ก-๙a-zA-Z]{2,15})", q
    )
    if m_real:
        nick, fname = m_real.group(1), m_real.group(2)
        mask = (
            (df_all[COL["nick_th"]] == nick) | (df_all[COL["nick_en"]].str.upper() == nick.upper())
        ) & (
            (df_all[COL["fname_th"]].str.contains(fname, na=False)) |
            (df_all[COL["fname_en"]].str.upper().str.contains(fname.upper(), na=False))
        )
        rows = df_all[mask]
        if not rows.empty:
            return _fmt_smart(rows.iloc[0], lang, _what_is_asked(q))

    # ── 2j. Nickname + branch — "แมว สาขาลาดพร้าว" ────────────────────────
    branch_code = next((c for kw, c in BRANCH_MAP.items() if kw.lower() in ql), None)
    if branch_code:
        m_nb = re.search(r"([ก-๙a-zA-Z\-]{1,10})\s+(?:สาขา|branch)", q, re.IGNORECASE)
        if m_nb:
            nick_b = m_nb.group(1).strip()
            mask = (
                (df_all[COL["nick_th"]] == nick_b) |
                (df_all[COL["nick_en"]].str.upper() == nick_b.upper()) |
                (df_all[COL["fname_th"]] == nick_b) |
                (df_all[COL["fname_en"]].str.upper() == nick_b.upper())
            ) & (df_all[COL["branch"]].str.contains(branch_code, na=False))
            rows = df_all[mask]
            if not rows.empty:
                return _fmt_smart(rows.iloc[0], lang, _what_is_asked(q))

    # ── 2j2. Bare nickname identity — "ใครคือปันปัน" / "who is PANPAN" ──
    m_nick_id = re.search(
        r"(?:ใครคือ\s*|who(?:'s| is)\s+)([ก-๙a-zA-Z\-]{2,12})\s*$",
        q.strip(), re.IGNORECASE,
    )
    if not m_nick_id:
        m_nick_id = re.search(r"^([ก-๙a-zA-Z\-]{2,12})\s*คือใคร$", q.strip(), re.IGNORECASE)
    if m_nick_id and not need_ext:
        rows = db.by_nickname(m_nick_id.group(1).strip())
        if len(rows) == 1:
            return _fmt_emp(rows.iloc[0], lang)
        if not rows.empty:
            return _fmt_list(rows.head(list_limit or 10), lang)

    # ── 2k. Nickname identity with dept anchor — "อรุณ ที่อยู่ SUP คือใคร" ─
    m_nick_who = re.search(
        r"([ก-๙a-zA-Z\-]{1,10})\s+(?:ที่อยู่|อยู่ใน|อยู่|จาก|from|in)\s+([A-Z]{2,5})\s*(?:คือใคร|คือ|ใคร|who)?",
        q, re.IGNORECASE,
    )
    if m_nick_who and not need_ext:
        row = _nick_dept_lookup(m_nick_who.group(1).strip(), m_nick_who.group(2).strip().upper())
        if row is not None:
            return _fmt_emp(row, lang)

    # ── 2l. Hierarchy queries ──────────────────────────────────────────────
    is_hierarchy = any(kw in ql for kw in [
        "รายงานตรง", "รายงานใคร", "ดูแลใคร", "reports to", "reports directly", "direct report"
    ])
    if is_hierarchy:
        units_h  = _get_unit_codes(q)
        target_u = units_h[0] if units_h else _resolve_unit_desc(q)

        if target_u == "CEO-CoS" or "ceo-cos" in ql or "cos" in ql:
            return _fmt_emp(db.by_unit("CEO").iloc[0], lang)

        if target_u:
            t_rows = db.by_unit(target_u)
            if not t_rows.empty:
                t_dept = t_rows.iloc[0][COL["dept"]]
                if target_u == "CPO":
                    subs = df_all[df_all[COL["unit"]].isin(["SFVP","DNVP","KSVP","WKVP","JCVP"])]
                    if not subs.empty:
                        return _fmt_list(subs, lang)
                sub_rows = df_all[
                    (df_all[COL["dept"]] == t_dept) &
                    (df_all[COL["level"]].isin(["VP", "Director"])) &
                    (df_all[COL["unit"]] != target_u)
                ]
                if not sub_rows.empty:
                    return _fmt_list(sub_rows.head(10), lang)

    # ── 2m. Multi-unit extension — "ext for SFVP, DNVP, KSVP" ────────────
    units = _get_unit_codes(q)
    if len(units) >= 2 and need_ext:
        parts = []
        for uc in units:
            r = db.by_unit(uc)
            if not r.empty:
                ro   = r.iloc[0]
                e_v  = ro[COL["ext"]]; m_v = ro[COL["mobile"]]
                if lang == "th":
                    line = f"{uc} — {_lang_name(ro, 'th')}: ต่อ {e_v}"
                    if m_v: line += f" / มือถือ {m_v}"
                else:
                    line = f"{uc} — {_lang_name(ro, 'en')}: ext {e_v}"
                    if m_v: line += f" / mobile {m_v}"
                parts.append(line)
        if parts:
            return "\n".join(parts)

    # ── 2n. Single explicit unit code — skip when counting (let dept handle) ─
    # Also skip when the token is a section code like CEO-OFF (handled by 2p).
    if units and not is_count and not (is_list and _get_dept_or_section(q) is not None):
        rows = db.by_unit(units[0])
        if not rows.empty:
            return (_fmt_emp(rows.iloc[0], lang)
                    if len(rows) == 1
                    else _fmt_list(rows, lang, inc_ext=need_ext))

    # ── 2o. Dept / Section listing or count — runs BEFORE unit_desc ──────
    # Must precede _resolve_unit_desc so that section codes like CEO-OFF are
    # not shadowed by the 'ceo' alias in UNIT_ALIAS.
    ds = _get_dept_or_section(q)
    if ds and (is_list or is_count):
        rows = (df_all[df_all[COL["section"]].str.upper() == ds]
                if ds in SECTION_SET
                else df_all[df_all[COL["dept"]].str.upper() == ds])
        if not rows.empty:
            if is_count:
                n = len(rows)
                return f"แผนก {ds} มี {n} คน" if lang == "th" else f"{ds} has {n} employees"
            return _fmt_list(rows.head(list_limit or 30), lang)

    # ── 2p. Description → unit (VP identity / informal role) ──────────────
    ud = _resolve_unit_desc(q)
    if ud:
        rows = db.by_unit(ud)
        if not rows.empty:
            return (_fmt_emp(rows.iloc[0], lang)
                    if len(rows) == 1
                    else _fmt_list(rows, lang, inc_ext=need_ext))

    # ── 2q. Size / count without explicit dept code ────────────────────────
    if is_count:
        dept_kw = {"ret":"RET","tec":"TEC","fin":"FIN","ops":"OPS","mkt":"MKT",
                   "hr":"HR","log":"LOG","sup":"SUP","ceo":"CEO","b2b":"B2B",
                   "sf":"SF","dn":"DN","ks":"KS","wk":"WK","jc":"JC","leg":"LEG"}
        for kw, dc in dept_kw.items():
            if re.search(r"\b" + kw + r"\b", ql):
                rows = df_all[df_all[COL["dept"]].str.upper() == dc]
                if not rows.empty:
                    n = len(rows)
                    return f"แผนก {dc} มี {n} คน" if lang == "th" else f"{dc} has {n} employees"

    # ── 2r. Org informal listing (subsidiary / retail brand names) ─────────
    for kw, dept in ORG_MAP.items():
        if kw.lower() in ql:
            rows = df_all[df_all[COL["dept"]].str.upper() == dept]
            if not rows.empty:
                if is_count:
                    n = len(rows)
                    return f"แผนก {dept} มี {n} คน" if lang == "th" else f"{dept} has {n} employees"
                return _fmt_list(rows.head(list_limit or 15), lang)

    # ── 2s. Thai knowledge — fruit / colour nickname sets ─────────────────
    if "ผลไม้" in ql or "fruit" in ql:
        rows = df_all[df_all[COL["nick_th"]].isin(FRUIT_NICK_TH) |
                      df_all[COL["nick_en"]].isin(FRUIT_NICK_EN)]
        if not rows.empty:
            return _fmt_list(rows.head(25), lang)

    if "ชื่อสี" in ql or "colour" in ql or "color" in ql:
        rows = df_all[df_all[COL["nick_th"]].isin(COLOR_NICK_TH) |
                      df_all[COL["nick_en"]].isin(COLOR_NICK_EN)]
        if not rows.empty:
            return _fmt_list(rows.head(25), lang)

    # ── 2t. Nickname grid / count ──────────────────────────────────────────
    nick_g_m = re.search(
        r"^([ก-๙a-zA-Z\-]{1,12})\s*(?:คือใคร|มีใครบ้าง|มีใคร|มีกี่คน|ใครบ้าง|who(?:\s+is)?)",
        q.strip(), re.IGNORECASE,
    )
    nick_g_m2 = re.search(
        r"(?:ชื่อเล่น|nickname)[^ก-๙a-zA-Z]*([ก-๙a-zA-Z\-]{1,12})",
        q.strip(), re.IGNORECASE,
    ) if not nick_g_m else None
    nick_g = (nick_g_m or nick_g_m2)
    if nick_g:
        nick_g = nick_g.group(1).strip()
        rows   = db.by_nickname(nick_g)
        if is_count or "กี่คน" in ql or "how many" in ql:
            n = len(rows)
            return (f"มีชื่อเล่น {nick_g} ทั้งหมด {n} คน" if lang == "th"
                    else f"There are {n} employees with nickname {nick_g}")
        if not rows.empty:
            return _fmt_list(rows, lang)
        return REFUSAL_PHRASES["person_not_found"][lang]

    # ── 2u. Branch geography knowledge ────────────────────────────────────
    geo = _resolve_geo(q, lang)
    if geo:
        return geo

    # ── 2v. Tier / level listing ───────────────────────────────────────────
    level_map = {
        "director":"Director","directors":"Director",
        "vp":"VP","vps":"VP","vice president":"VP",
        "manager":"Manager","managers":"Manager",
        "lead":"Lead","c-level":"C-level",
    }
    for kw, level in level_map.items():
        if kw in ql and (is_list or is_count or "ทั้งหมด" in ql or "all" in ql):
            rows = df_all[df_all[COL["level"]] == level]
            if not rows.empty:
                if is_count:
                    n = len(rows)
                    return (f"มี {level} ทั้งหมด {n} คน" if lang == "th"
                            else f"There are {n} {level}s")
                return _fmt_list(rows.head(40), lang)

    # ── 2w. Surname / family count ─────────────────────────────────────────
    if "นามสกุล" in ql or "surname" in ql or "ญาติ" in ql or "family" in ql:
        m_sn = re.search(r"(?:นามสกุล|surname)\s+([\u0E00-\u0E7Fa-zA-Z]+)", q, re.IGNORECASE)
        if m_sn:
            sn   = m_sn.group(1)
            rows = df_all[
                (df_all[COL["lname_th"]] == sn) |
                (df_all[COL["lname_en"]].str.upper() == sn.upper())
            ]
            if rows.empty and not is_count:
                rows = df_all[df_all[COL["lname_th"]].str.contains(sn, na=False) |
                              df_all[COL["lname_en"]].str.upper().str.contains(sn.upper(), na=False)]
            if is_count:
                n = len(rows)
                return (f"มีพนักงานนามสกุล {sn} ทั้งหมด {n} คน" if lang == "th"
                        else f"There are {n} employees with surname {sn}")
            if not rows.empty:
                return _fmt_list(rows, lang)
        if "ญาติ" in ql or "family" in ql:
            sn_count = Counter(df_all[COL["lname_th"]].tolist())
            common   = [s for s, c in sn_count.most_common(5) if c >= 2 and s]
            if lang == "th":
                return "ตัวอย่างนามสกุลที่พบหลายคนในองค์กร: " + ", ".join(common)
            return "Example surnames shared by multiple employees: " + ", ".join(common)

    # ── 2w2. Count employees by first name or nickname ────────────────────
    m_name_count = re.search(
        r"(?:นับคนชื่อ|กี่คนชื่อ|count(?: employees)? named|how many(?: employees)? (?:named|called))\s*([ก-๙a-zA-Z\-]{2,15})",
        q, re.IGNORECASE,
    )
    if m_name_count:
        token = m_name_count.group(1).strip()
        token = re.sub(r"(?:ให้หน่อย|หน่อย|ให้|นะ|ครับ|ค่ะ)$", "", token).strip()
        rows = df_all[
            (df_all[COL["fname_th"]] == token) |
            (df_all[COL["fname_en"]].str.upper() == token.upper()) |
            (df_all[COL["nick_th"]] == token) |
            (df_all[COL["nick_en"]].str.upper() == token.upper())
        ]
        n = len(rows)
        return (f"มีทั้งหมด {n} คน" if lang == "th"
                else f"There are {n} employees")

    # ── 2x. First name listing — "ใครชื่อไพบูลย์" ────────────────────────
    for pat in [r"(?:ใครชื่อ|คนชื่อ|รายชื่อคนชื่อ)\s*([ก-๙a-zA-Z]{2,15})",
                r"(?:who(?:'s| is) named|employees? named?)\s+([a-zA-Z]{2,15})"]:
        m_fn = re.search(pat, q, re.IGNORECASE)
        if m_fn:
            fname = m_fn.group(1)
            mask  = (df_all[COL["fname_th"]].str.contains(fname, na=False) |
                     df_all[COL["fname_en"]].str.upper().str.contains(fname.upper(), na=False))
            rows  = df_all[mask]
            if not rows.empty:
                return _fmt_list(rows.head(list_limit or 20), lang)

    # ── 2y. Non-listing dept/section fallback (e.g. just "TEC-EXEC") ──────
    if ds:
        rows = (df_all[df_all[COL["section"]].str.upper() == ds]
                if ds in SECTION_SET
                else df_all[df_all[COL["dept"]].str.upper() == ds])
        if not rows.empty:
            return _fmt_list(rows.head(list_limit or 20), lang)

    return None  # Stage 2 miss → proceed to Stage 3

# ═════════════════════════════════════════════════════════════════════════════

# Stage 4: LLM Contextual Answerer
### **การสรุปผลและตอบคำถาม:** นำข้อมูลจริง (True Content) ที่ดึงได้จาก Stage 3 มาให้ Typhoon ทำการสรุปเป็นคำตอบที่สั้น กระชับ และตรงประเด็นตามที่ผู้ใช้ถาม โดยมีการควบคุมรูปแบบภาษาให้ถูกต้องตามคำถามต้นฉบับ

In [9]:
def stage4_answerer(q: str, lang: str, ctx: str) -> str:
    """Stage 4 — Ask Typhoon to answer the question given the retrieved context."""
    raw = _llm_call([
        {"role": "system", "content": _ANSWERER_SYS[lang]},
        {"role": "user", "content": f"Employee data:\n{ctx}\n\nQuestion: {q}"},
    ])
    return raw

# ═════════════════════════════════════════════════════════════════════════════

# System Orchestrator: Integration Layer
### **ตัวควบคุมการทำงานทั้งระบบ:** ทำหน้าที่จัดการลำดับขั้น (Pipeline Orchestration) ตั้งแต่ Stage 1 ถึง Stage 5 เพื่อให้การไหลของข้อมูลมีความต่อเนื่องและสามารถจัดการกับเคสข้อยกเว้นต่างๆ ได้อย่างเป็นระบบ

In [10]:
# Orchestrator — ties all 5 stages together
# ═════════════════════════════════════════════════════════════════════════════

def answer_question(qid: str, lang: str, question: str) -> str:
    """
    Orchestrator — ties all 5 stages together.
    Every exit path passes through Stage 5 (compliance_guard).

    Flow:
      injection strip → Stage 1 (refusal?) → Stage 2 (fast-path hit?) →
      Stage 3 (Typhoon query) → Stage 4 (Typhoon answer) → Stage 5 (guard)
      → fallback: person_not_found
    """
    q  = question.strip()
    ql = q.lower()

    # ── Injection pre-processing: strip preamble, re-run on real question ──
    # Pure injection with no embedded real question falls through to Stage 1.
    embedded_q = _extract_embedded_question(q)
    if embedded_q:
        return answer_question(qid, lang, embedded_q)

    # ── Stage 1 — Intent Router ───────────────────────────────────────────
    refusal = stage1_intent_router(q, lang)
    if refusal is not None:
        # Refusal answers must never leak phone extensions or employee IDs
        return compliance_guard(refusal, lang, is_refusal=True)

    # ── Stage 2 — Deterministic Fast-Path ────────────────────────────────
    fast = stage2_fast_path(q, lang)
    if fast is not None:
        # Data answers may legitimately contain phone extensions (contact lookups)
        return compliance_guard(fast, lang, is_refusal=False)

    # ── Stage 3 — Typhoon Planner ─────────────────────────────────────────
    query_str = stage3_planner(q)
    if query_str:
        df_res = db.execute_query(query_str)
        if not df_res.empty:
            # ── Stage 4 — Typhoon Answerer ────────────────────────────────
            ctx = _rows_to_ctx(df_res)
            ans = stage4_answerer(q, lang, ctx)
            if ans and len(ans) > 3:
                return compliance_guard(ans, lang, is_refusal=False)
        # Stage 3 returned empty rows or Stage 4 returned nothing useful
        # → fall through to canonical person_not_found

    # ── Stage 5 fallback ─────────────────────────────────────────────────
    return REFUSAL_PHRASES["person_not_found"][lang]

# ═════════════════════════════════════════════════════════════════════════════

# Execution Engine: Batch Processing
### **ระบบประมวลผลคำถามจำนวนมาก:** รันคำถามทั้ง 300 ข้อแบบ Batch พร้อมระบบ Error Handling และ Logging เพื่อให้ได้ผลลัพธ์ที่เสถียรที่สุด

In [11]:
# Main loop
# ═════════════════════════════════════════════════════════════════════════════

def main() -> None:
    print("=" * 60)
    print("FahMai Directory Q&A — 5-Stage Pipeline")
    print(f"  Stage 1/2: deterministic (regex + Pandas)")
    print(f"  Stage 3/4: Typhoon ({TYPHOON_MODEL})")
    print(f"  Stage 5:   compliance guard")
    print("=" * 60)

    questions: list[dict] = []
    with open(QUESTIONS_CSV, encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            questions.append(row)
    print(f"Loaded {len(questions)} questions.\n")

    results: list[dict] = []
    for i, row in enumerate(questions, 1):
        qid  = row["id"].strip()
        lang = row["language"].strip()
        q    = row["question"].strip()
        print(f"[{i:03d}/{len(questions)}] {qid} ({lang}) {q[:65]}")
        try:
            ans = answer_question(qid, lang, q)
        except Exception as exc:
            print(f"  !! {exc!r}")
            ans = REFUSAL_PHRASES["person_not_found"].get(lang, "no record found")
        print(f"  → {ans[:110]}")
        results.append({"id": qid, "response": ans})
        time.sleep(0.15)

    with open(SUBMISSION_CSV, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["id", "response"])
        w.writeheader()
        w.writerows(results)

    print(f"\n✅ Saved {len(results)} rows → {SUBMISSION_CSV}")
    print("Grade: python grade.py submission.csv train_labels.json")


if __name__ == "__main__":
    if TYPHOON_API_KEY == "YOUR_TYPHOON_API_KEY_HERE":
        print("⚠️  export TYPHOON_API_KEY=your_key_here")
        sys.exit(1)
    main()

FahMai Directory Q&A — 5-Stage Pipeline
  Stage 1/2: deterministic (regex + Pandas)
  Stage 3/4: Typhoon (typhoon-v2.5-30b-a3b-instruct)
  Stage 5:   compliance guard
Loaded 300 questions.

[001/300] g001 (en) who is the RETVP
  → WIRIYA CHANCHAI (nickname: TIK) | VICE PRESIDENT RETAIL NETWORK
[002/300] g002 (th) ใครเป็น OPSVP
  → คึกฤทธิ์ บุษราคัมวงศ์ | รองประธานฝ่ายปฏิบัติการ
[003/300] g004 (th) LEGVP ใคร
  → ไพโรจน์ มหากุล | รองประธานฝ่ายกฎหมาย
[004/300] g005 (th) ใครเป็น COO ตอนนี้
  → พงษ์กานต์ ราชชากัญญ์ | ประธานเจ้าหน้าที่ปฏิบัติการ
[005/300] g007 (th) ใครเป็น LOGFL ตอนนี้
  → มาลี อมรทอง ชื่อเล่น คลาวด์ | รองประธานฝ่ายขนส่ง
[006/300] g008 (th) ใครเป็น DNVP ตอนนี้
  → เรืองศักดิ์ เทพเกียรติกำจร | รองประธานฝ่ายดาวเหนือ
[007/300] g009 (th) SUPVP ใคร
  → ดาริกา อาวุทธ์ดี ชื่อเล่น ตูน | รองประธานฝ่ายบริการลูกค้า
[008/300] g011 (en) who is the CHRO
  → NATHAMON APHICHAIDEE | CHIEF HUMAN RESOURCES OFFICER
[009/300] g012 (th) ใครเป็น MKTVP
  → คะวัง กอบสุขรัตน์ ชื่อเล่น โอ | รองประธานฝ